# `# @cash:cache-calls` — try it yourself

Cash normally caches a **statement**. When the statement is a cheap wrapper around
an expensive call, that unit is wrong:

| statement | what cash does by default | why |
|---|---|---|
| `out.append(compute(x))` | never reuses anything | the append mutates an object that already exists, so there is no snapshot to restore |
| `s += compute(x)` | reuses only an unchanged *prefix* | each iteration reads `s`, so its key encodes every iteration before it — reorder the list and the tail re-runs |

`# @cash:cache-calls` moves the cache one level down, onto `compute(x)` itself.
The cheap wrapper still executes every run (so the append really happens); the
expensive call is served from cache.

**How to use this notebook:** run the cells top to bottom once, then go back and
**re-run individual cells** as each section tells you. The counter and the badge
are the evidence — not the wall clock, which cannot tell "recomputed" from
"restored but slow".

In [ ]:
import cash
%cash_on

## Setup

`compute()` sleeps for a second and records that it ran. `CALLS` is the ground
truth throughout: **if a number appears in `CALLS`, that call really executed.**

> You will see a `CashImpurityWarning` about `CALLS.append()` the first time a
> cached call runs. That is cash being correct — appending to a global *is* a
> side effect, and cash is telling you a cached call will skip it. Here the side
> effect is only the counter, so it is safe to ignore; in real code it is worth
> reading.

In [ ]:
import time

CALLS = []

def compute(x):
    CALLS.append(x)      # ground truth: this line only runs on a real execution
    time.sleep(1.0)      # stand-in for real work
    return x + 1

def merge(a, b):
    return a + b

def show(label=""):
    print(f"{label:<22} CALLS = {CALLS}   ({len(CALLS)} real executions so far)")

show("after setup")

---
## 1. An append loop — the shape that never reuses anything

The list is created in **its own cell**, and that detail is load-bearing. When
`out = []` sits in the *same* cell as the loop, cash can cache the whole cell as
one unit and the problem disappears. The interesting — and much more common —
case is a container built earlier and appended to later.

Run the next cell once, then re-run **only the loop cell after it**, twice.
Watch `CALLS` grow every time: three more real executions on every run, for
ever. The badge says `NOT CACHED · In-place mutation on: out`.

In [ ]:
out = []          # built HERE, appended to in the next cell

In [ ]:
for x in [1, 2, 3]:
    out.append(compute(x))

show("append, no directive")

### Now with the directive

Same shape, one comment added. Run the `out2 = []` cell once, then re-run the
loop cell **twice**.

The first run adds three executions (nothing is cached yet). **The second run
adds none** — and `out2` is still correct, because the append itself still runs.

On the badge, open the row and look for:

```
compute() [via @cash:cache-calls]: 3/3 cached
```

That tag is how you confirm the directive engaged. `@cache-calls` rather than
`@cache` means *cash* wrapped the call, not you.

In [ ]:
out2 = []         # again, built in its own cell

In [ ]:
# @cash:cache-calls
for x in [1, 2, 3]:
    out2.append(compute(x))

show("append + directive")
print("out2 =", out2)

---
## 2. Reordering a loop — the prefix problem

This is the one that motivated the feature.

Run the next cell once. Then **change the list to `[3, 2, 1]`** and run it again.

Without the directive, reuse is a *prefix* property: the accumulator `s` makes
each iteration depend on every iteration before it, so changing the **first**
element re-runs everything. Appending to the end is free; reordering is not.

In [ ]:
s = 0
for x in [10, 20, 30]:      # <-- try [30, 20, 10] on the second run
    s += compute(x)

print("SUM", s)
show("fold, no directive")

### The same fold, with the directive

Run once, then **reorder the list any way you like** and run again.

The statement-level entries still miss — the loop genuinely re-executes — but
every `compute(x)` hits its own cache entry, so **no new executions appear**.
A call cache keys on arguments, not on execution history, so it is
order-independent by construction.

Then try adding a genuinely new value (say `40`): exactly one new execution.

In [ ]:
s2 = 0
# @cash:cache-calls
for x in [11, 22, 33]:      # <-- reorder freely; then try adding 44
    s2 += compute(x)

print("SUM", s2)
show("fold + directive")

---
## 3. When the directive can't help — and says so

A call is only extractable when it does **not** read the statement's own
assignment or mutation target. If it does, it *is* the fold and there is no
order-independent value to pull out of it.

| statement | cached call |
|---|---|
| `s += compute(x)` | `compute(x)` |
| `out.append(compute(x))` | `compute(x)` |
| `prices[k] = compute(k)` | `compute(k)` |
| `s = merge(s, x)` | none — the call reads `s` |
| `df.sort_values(inplace=True)` | none — the mutation *is* the work |

The next cell asks for something impossible. Rather than silently doing nothing,
cash raises a `CashCacheIneffectiveWarning` telling you why — once per statement,
not once per iteration.

In [ ]:
acc = 0
# @cash:cache-calls
acc = merge(acc, 5)      # `merge` reads `acc`, the target -> nothing to extract

print("acc =", acc)

---
## What to look for

- **`CALLS`** is ground truth. A number appears only when the work really ran.
- **The badge tag** `compute() [via @cash:cache-calls]` confirms the directive
  engaged. Plain `@cache` means you decorated that function yourself.
- **A warning** means the directive matched nothing — check whether your call
  reads the statement's target.

### What is deliberately *not* intercepted

- **Already-decorated functions.** They are on this path already; wrapping them
  again would split their hits across two cache entries.
- **Builtins.** A hot loop must not pay for a cache key per `len()`.
- **Bound methods** (`model.predict(x)`). Caching a method puts `self` in the
  key, which needs your judgement rather than cash's guess — see
  [caching class methods](https://cash-lib.readthedocs.io/en/latest/tutorials/feature-guides/caching-class-methods/).
  Decorate the method yourself when you want that.

### Reference

- [`# @cash:cache-calls`](https://cash-lib.readthedocs.io/en/latest/annotations/#cashcache-calls-alias-cachecalls)
- [Reordering a loop's items](https://cash-lib.readthedocs.io/en/latest/known-limitations/#reordering-a-loops-items-re-runs-the-tail)

> **Placement matters.** A `@cash:` directive attaches to the statement *below*
> it, and the backwards scan stops at the first non-comment line. On a loop, put
> it on the `for` header — on the cell's first line it would scope to whatever
> statement happens to be first.